In [ ]:
import os
from pathlib import Path
from glob import glob

import torch
import numpy as np

from gait_ml.model.litmodel import LitSeq2Seq
from gait_ml.data.dataset import GaitDataset
from gait_ml import evaluate
from gait_ml import plotting
from gait_ml.data.datamodule import GaitDataModule

%matplotlib inline
%load_ext autoreload
%autoreload 2

### 1. Predict gait event labels given a list of files

# Load model
model_fpath = "/home/qivy00li/projects/gait_ml/backpain/ZscaledRerunExp4-Fold1-GRU-expandlabel2_2025-11-10_15-13-18/checkpoints/model-epoch=97-val_f1score=0.88.ckpt"
device_ = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
best_model = LitSeq2Seq.load_from_checkpoint(model_fpath).to(device_)
model = best_model.model

# Load dataset
files = glob("../data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_01_T1/*_2mW_IPhone.xls")
files

zscale_stats =  {'mean': np.array([-0.26654707, -0.02442654,  0.37051307, -0.00488071,  1.36632781, 0.86179046]), 
                'std': np.array([1.42070944, 2.89440634, 1.53666348, 5.25694552, 3.22176259,5.12151813]),}

test_datamodule = GaitDataModule(files,
     batch_size=32,
     window_size=256,
     step_size=256,
     train_idx=None, 
     val_idx=None, # We load validation/test set via the test_idx here
     test_idx=[0],
     expand_labels=0,
     acc_sheet_name="Linear Accelerometer",
     num_workers=1,
     zscale=True,
     zscale_stats=zscale_stats,
)
cur_data = GaitDataset(
    csv_files=files,
    window_size=256,
    step_size=256,
    expand_labels=0,
    acc_sheet_name="Linear Accelerometer",
    gyr_sheet_name="Gyroscope",
    zscale=True,
    zscale_stats=zscale_stats,
)
test_datamodule.test_dataset = cur_data
test_dataloader = test_datamodule.test_dataloader()

best_model.eval()
device = best_model.device

with torch.no_grad():
    cur_pred = []
    for i in test_dataloader:
        # cur_res = best_model.model(i[0].to(device), i[1].to(device))
        cur_res = best_model(i[0].to(device), i[1].to(device))
        cur_pred.append(cur_res.cpu())
    cur_pred = torch.concat(cur_pred).reshape(-1, 3)
    cur_pred = cur_pred.argmax(-1).numpy()

    # Select only the middle prediction as the final event
    merged_preds = evaluate.merge_clustered_events(cur_pred)

    # Plot first 2 seconds
    debug = False
    sec_to_plot = 2000
    if debug:
        raw_x = cur_data.raw_x[0][:sec_to_plot]
        raw_y = cur_data.raw_y[0][:sec_to_plot]
        gt_labels_fig = plotting.plot_xyz(raw_x[:, 0], raw_x[:, 1], raw_x[:, 2], labels=raw_y[:sec_to_plot])
        gt_labels_fig.show()
        pred_labels_fig = plotting.plot_xyz(raw_x[:, 0], raw_x[:, 1], raw_x[:, 2], labels=cur_pred[:sec_to_plot])
        pred_labels_fig.show()
        merge_pred_fig = plotting.plot_xyz(raw_x[:, 0], raw_x[:, 1], raw_x[:, 2], labels=merged_preds[:sec_to_plot])
        merge_pred_fig.show()


## Test programmatic inference

In [142]:
from gait_ml.predict import predict_gait_event_labels
from glob import glob

files = glob("../data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_01_T1/*_2mW_IPhone.xls")
model_fpath = "/home/qivy00li/projects/gait_ml/backpain/ZscaledRerunExp4-Fold1-GRU-expandlabel2_2025-11-10_15-13-18/checkpoints/model-epoch=97-val_f1score=0.88.ckpt"
predict_gait_event_labels(model_fpath, files)

../data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_01_T1/01_1_2mW_IPhone.xls
cropped_array shape: (53, 256, 7)
all_arrays shape: (53, 256, 7)


array([0, 0, 0, ..., 0, 0, 0], shape=(13568,))